# Forward Kinematics Propagation Experiment

This experiment tests how parent bone rotations propagate down to children (both connected and unconnected) in Blender's armature system, and verifies the mathematical formulas for forward kinematics.

In [8]:
import sys
import os

# Ensure workspace root is in path
if ".." not in sys.path:
    sys.path.append("..")

from animgen.core.armature import Armature, Bone
from animgen.utils.math import rotation_matrix_from_vectors
import bpy

import numpy as np


In [9]:
def clear_scene():
    """Clear all objects and orphaned data blocks from the scene."""
    if bpy.context.active_object and bpy.context.active_object.mode != 'OBJECT':
        bpy.ops.object.mode_set(mode='OBJECT')
    bpy.ops.object.select_all(action="SELECT")
    bpy.ops.object.delete()

    for collection in (
        bpy.data.meshes,
        bpy.data.armatures,
        bpy.data.materials,
        bpy.data.cameras,
        bpy.data.lights,
        bpy.data.images,
    ):
        for block in collection:
            if block.users == 0:
                collection.remove(block)

In [10]:
def create_blender_armature(armature_data):
    """Reconstruct the custom Armature inside Blender in EDIT mode."""
    arm_data = bpy.data.armatures.new("FKTestArmature")
    arm_obj = bpy.data.objects.new("FKTestArmature", arm_data)

    bpy.context.collection.objects.link(arm_obj)
    bpy.context.view_layer.objects.active = arm_obj
    bpy.ops.object.mode_set(mode="EDIT")

    edit_bones = {}

    def create_bone_recursive(custom_bone):
        eb = arm_data.edit_bones.new(custom_bone.id)
        eb.head = custom_bone.head
        eb.tail = custom_bone.tail
        edit_bones[custom_bone.id] = eb

        if custom_bone.parent is not None:
            eb.parent = edit_bones[custom_bone.parent.id]
            eb.use_connect = custom_bone.is_connected_to_parent

        for child in custom_bone.child:
            create_bone_recursive(child)

    create_bone_recursive(armature_data.root_bone)

    bpy.ops.object.mode_set(mode="OBJECT")
    return arm_obj

## Step 1: Define armature structure
We create a straight connected bone chain along the Z-axis, plus an unconnected offset bone.

In [11]:
clear_scene()

# Root bone
root = Bone(id="root", head=(0, 0, 0), tail=(0, 0, 1))
armature = Armature(root)

# Connected children
bone_1 = armature.add_connected_bone(parent=root, tail=(0, 0, 2))
bone_2 = armature.add_connected_bone(parent=bone_1, tail=(0, 0, 3))

# Unconnected offset child of bone_1
bone_3 = armature.add_unconnected_bone(parent=bone_1, head=(1, 0, 2), tail=(2, 0, 2))

arm_obj = create_blender_armature(armature)

print("Edit Mode Bones:")
for b in arm_obj.data.bones:
    print(f"Bone {b.name}: head={list(b.head_local)}, tail={list(b.tail_local)}, connected={b.use_connect}")

Edit Mode Bones:
Bone root: head=[0.0, 0.0, 0.0], tail=[0.0, 0.0, 1.0], connected=False
Bone 19ee1dc7-c3bc-478f-9724-8be41a70e49f: head=[0.0, 0.0, 1.0], tail=[0.0, 0.0, 2.0], connected=True
Bone 531f2805-120e-4e7e-8e72-a255318cd842: head=[0.0, 0.0, 2.0], tail=[0.0, 0.0, 3.0], connected=True
Bone e71cbd9e-1035-431d-b952-297497d4f667: head=[1.0, 0.0, 2.0], tail=[2.0, 0.0, 2.0], connected=False


## Step 2: Apply rotation and check propagation

In [12]:
bpy.ops.object.mode_set(mode="POSE")

pb_1 = arm_obj.pose.bones[bone_1.id]
# Rotate bone_1 around X-axis by 45 degrees
pb_1.rotation_mode = "XYZ"
pb_1.rotation_euler = (np.pi / 4.0, 0.0, 0.0)

# Update view layer / depgraph
bpy.context.view_layer.update()

print("Pose Mode World Coordinates:")
for pb in arm_obj.pose.bones:
    print(f"Bone {pb.name}: head={list(pb.head)}, tail={list(pb.tail)}")

Pose Mode World Coordinates:
Bone root: head=[0.0, 0.0, 0.0], tail=[0.0, 0.0, 1.0]
Bone 19ee1dc7-c3bc-478f-9724-8be41a70e49f: head=[0.0, 0.0, 1.0], tail=[0.0, -0.7071067690849304, 1.7071068286895752]
Bone 531f2805-120e-4e7e-8e72-a255318cd842: head=[0.0, -0.7071067690849304, 1.7071068286895752], tail=[0.0, -1.4142135381698608, 2.4142136573791504]
Bone e71cbd9e-1035-431d-b952-297497d4f667: head=[1.0, -0.7071067690849304, 1.7071068286895752], tail=[2.0, -0.7071067690849304, 1.7071068286895752]


## Step 3: Verify the "Invisible Helper Bone" Equivalence

We will now verify if we can replace the unconnected offset bone (`bone_3`) with an equivalent setup using a virtual connected bone.
We define a virtual armature where:
- `v_invisible` is an unconnected bone parented to `v_bone_1` spanning from the pivot head `(0, 0, 1)` to the child's head `(1, 0, 2)`.
- `v_bone_3` is a connected child of `v_invisible`, starting at `(1, 0, 2)` and ending at `(2, 0, 2)`.

We then apply the same rotation and compare coordinates.

In [13]:
clear_scene()

v_root = Bone(id="v_root", head=(0, 0, 0), tail=(0, 0, 1))
v_armature = Armature(v_root)

# Connected chain
v_bone_1 = v_armature.add_connected_bone(parent=v_root, tail=(0, 0, 2))
v_bone_2 = v_armature.add_connected_bone(parent=v_bone_1, tail=(0, 0, 3))

# Virtual invisible bone parented to v_bone_1
v_invisible = v_armature.add_unconnected_bone(parent=v_bone_1, head=(0, 0, 1), tail=(1, 0, 2))

# v_bone_3 connected to the end of the invisible bone
v_bone_3 = v_armature.add_connected_bone(parent=v_invisible, tail=(2, 0, 2))

v_arm_obj = create_blender_armature(v_armature)

bpy.ops.object.mode_set(mode="POSE")

v_pb_1 = v_arm_obj.pose.bones[v_bone_1.id]
v_pb_1.rotation_mode = "XYZ"
v_pb_1.rotation_euler = (np.pi / 4.0, 0.0, 0.0)

bpy.context.view_layer.update()

print("Virtual Pose Mode World Coordinates:")
for pb in v_arm_obj.pose.bones:
    print(f"Bone {pb.name}: head={list(pb.head)}, tail={list(pb.tail)}")

bpy.ops.object.mode_set(mode="OBJECT")

Virtual Pose Mode World Coordinates:
Bone v_root: head=[0.0, 0.0, 0.0], tail=[0.0, 0.0, 1.0]
Bone 9e8450d0-044a-4d02-970d-155838aa500a: head=[0.0, 0.0, 1.0], tail=[0.0, -0.7071067690849304, 1.7071068286895752]
Bone ab4f6793-b71e-4e0c-acec-c8b70c7ab545: head=[0.0, -0.7071067690849304, 1.7071068286895752], tail=[0.0, -1.4142135381698608, 2.4142136573791504]
Bone c24b0567-a1bc-488e-a6ec-82f23ec6705e: head=[0.0, 0.0, 1.0], tail=[0.9999998807907104, -0.7071067094802856, 1.7071067094802856]
Bone 7154c9ce-4722-4d7d-8677-c068ba8c7adf: head=[0.9999998807907104, -0.7071067094802856, 1.7071067094802856], tail=[1.9999998807907104, -0.7071068286895752, 1.7071067094802856]


{'FINISHED'}

## Step 4: Verify Single Rotation Matrix T Transformation for Hierarchy

Here, we verify that the rotation matrix $T$ that transforms the parent bone's vector is also the **exact transformation** for the entire downstream hierarchy (both connected and unconnected child bones) using the parent's head as the pivot center:

$$\vec{p}_{\text{pose}, c} = \vec{h}_{\text{pose}, p} + T \cdot (\vec{p}_{\text{edit}, c} - \vec{h}_{\text{edit}, p})$$

Let's test this in code using the `rotation_matrix_from_vectors` utility.

In [14]:
# Re-evaluate the original armature first
clear_scene()
root = Bone(id="root", head=(0, 0, 0), tail=(0, 0, 1))
armature = Armature(root)
bone_1 = armature.add_connected_bone(parent=root, tail=(0, 0, 2))
bone_2 = armature.add_connected_bone(parent=bone_1, tail=(0, 0, 3))
bone_3 = armature.add_unconnected_bone(parent=bone_1, head=(1, 0, 2), tail=(2, 0, 2))

arm_obj = create_blender_armature(armature)

bpy.ops.object.mode_set(mode="POSE")
pb_1 = arm_obj.pose.bones[bone_1.id]
pb_1.rotation_mode = "XYZ"
pb_1.rotation_euler = (np.pi / 4.0, 0.0, 0.0)
bpy.context.view_layer.update()

# 1. Define Rest vectors and Head pivot
h_edit_1 = np.array([0.0, 0.0, 1.0])
t_edit_1 = np.array([0.0, 0.0, 2.0])
A_parent = t_edit_1 - h_edit_1  # Parent rest vector: (0, 0, 1)

h_pose_1 = np.array(pb_1.head)  # (0, 0, 1)
t_pose_1 = np.array(pb_1.tail)  # (0, -0.7071, 1.7071)
B_parent = t_pose_1 - h_pose_1  # Parent posed vector

# 2. Calculate the rotation matrix T that aligns A_parent to B_parent
T = rotation_matrix_from_vectors(A_parent, B_parent)

# 3. Apply the single matrix T to all children joints, pivoting around the parent head
def transform_point(p_edit):
    return h_pose_1 + T @ (np.array(p_edit) - h_edit_1)

# Rest coordinates of children
h_edit_2 = np.array([0.0, 0.0, 2.0])
t_edit_2 = np.array([0.0, 0.0, 3.0])

h_edit_3 = np.array([1.0, 0.0, 2.0])
t_edit_3 = np.array([2.0, 0.0, 2.0])

# Compute manual positions
h_pose_2_calc = transform_point(h_edit_2)
t_pose_2_calc = transform_point(t_edit_2)

h_pose_3_calc = transform_point(h_edit_3)
t_pose_3_calc = transform_point(t_edit_3)

# 4. Get actual Blender coordinates
pb_2 = arm_obj.pose.bones[bone_2.id]
pb_3 = arm_obj.pose.bones[bone_3.id]

# 5. Assert equality
np.testing.assert_allclose(h_pose_2_calc, pb_2.head, atol=1e-6)
np.testing.assert_allclose(t_pose_2_calc, pb_2.tail, atol=1e-6)

np.testing.assert_allclose(h_pose_3_calc, pb_3.head, atol=1e-6)
np.testing.assert_allclose(t_pose_3_calc, pb_3.tail, atol=1e-6)

print("Parent World rotation matrix T:\n", T)
print("\nCalculated Coordinates:")
print(f"  bone_2: head={h_pose_2_calc.tolist()}, tail={t_pose_2_calc.tolist()}")
print(f"  bone_3: head={h_pose_3_calc.tolist()}, tail={t_pose_3_calc.tolist()}")
print("\nSUCCESS: Rotation matrix T is sufficient to transform the entire child hierarchy!")

bpy.ops.object.mode_set(mode="OBJECT")

Parent World rotation matrix T:
 [[ 1.          0.          0.        ]
 [ 0.          0.70710681 -0.70710675]
 [ 0.          0.70710675  0.70710681]]

Calculated Coordinates:
  bone_2: head=[0.0, -0.7071067513842252, 1.7071068109888685], tail=[0.0, -1.4142135027684504, 2.414213621977737]
  bone_3: head=[1.0, -0.7071067513842252, 1.7071068109888685], tail=[2.0, -0.7071067513842252, 1.7071068109888685]

SUCCESS: Rotation matrix T is sufficient to transform the entire child hierarchy!


{'FINISHED'}

## Conclusion & Key Takeaways

The mathematical verification and virtual bone setup demonstrate three fundamental properties of Forward Kinematics (FK):

### 1. The "Invisible Helper Bone" Equivalence
An unconnected offset bone (like `bone_3`, which has a parent-child relationship but is not physically connected head-to-tail) is mathematically equivalent to:
* Inserting an **invisible, rigid virtual bone** that starts at the parent's head and ends at the child's head.
* Parenting the child bone to the end of this virtual bone with a connected relationship (`use_connect = True`).

Both setups yield the exact same world-space pose coordinates because the offset vector behaves like a rigid extension of the parent's coordinate frame.

### 2. Parent Rotation Sufficiency (Subtree Rigidity)
When a parent bone rotates by a transformation $T$, all joints in its subtree (both connected and unconnected children) are transformed by the **exact same rotation matrix $T$** pivoting around the parent's head:
$$\vec{p}_{\text{pose}, c} = \vec{h}_{\text{pose}, p} + T \cdot (\vec{p}_{\text{edit}, c} - \vec{h}_{\text{edit}, p})$$

### 3. Local/Relative Animation Simplification
Because of this hierarchical propagation, you only need to define the **relative (local) rotation $L_i$** for each bone with respect to its parent:
$$W_c = W_p \cdot \text{relative\_offset}_c \cdot L_c$$

The system automatically resolves:
*snapped connectivity* (for connected bones) and *rotated offset translations* (for unconnected bones) down the entire hierarchy, meaning you only manipulate relative rotations to animate the entire creature.

### 4. Rotation Transformation T
Essentially if we move an object by T 3x3 rotation matrix this would mean that all its children whether connected or disconnected would also move by rotation matrix T wrt the head of the bone/parent where we started rotating which acts as the pivot